In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.tools import tool, ToolRuntime

@tool
def read_email(runtime: ToolRuntime) -> str:
    """Read an email from the given address."""
    # take email from state
    return runtime.state["email"]

@tool
def send_email(body: str) -> str:
    """Send an email to the given address with the given subject and body."""
    # fake email sending
    return f"Email sent"

In [3]:
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

class EmailState(AgentState):
    email: str

agent = create_agent(
    model="ollama:qwen2.5:3b",
    tools=[read_email, send_email],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False,
                "send_email": True,
            },
            description_prefix="Tool execution requires approval",
        ),
    ],
)

In [5]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {
        "messages": [HumanMessage(content="Please read my email and send a response immediately. Send the reply now in the same thread.")],
        "email": "Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John."
    },# type: ignore
    config=config # type: ignore
)

In [6]:
from pprint import pprint

pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi '
                                                                          'John, '
                                                                          'Thank '
                                                                          'you '
                                                                          'for '
                                                                          'letting '
                                                                          'me '
                                                                          'know '
                                                                          'about '
                                                                          'your '
                                                                          'delay. '
                                                                          "I'm "
               

In [7]:
print(response['__interrupt__'])

[Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'to': 'John@example.com', 'subject': 'Re: Meeting Rescheduling Request', 'body': "Hi John, Thank you for letting me know about your delay. I'm fine with rescheduling the meeting to tomorrow afternoon if that works better for you. Best regards."}, 'description': 'Tool execution requires approval\n\nTool: send_email\nArgs: {\'to\': \'John@example.com\', \'subject\': \'Re: Meeting Rescheduling Request\', \'body\': "Hi John, Thank you for letting me know about your delay. I\'m fine with rescheduling the meeting to tomorrow afternoon if that works better for you. Best regards."}'}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]}, id='034fe414ec9cc331fd61c97b2029e962')]


In [8]:
# Access just the 'body' argument from the tool call
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Hi John, Thank you for letting me know about your delay. I'm fine with rescheduling the meeting to tomorrow afternoon if that works better for you. Best regards.


# Approve

In [9]:
from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}
    ), 
    config=config # type:ignore # Same thread ID to resume the paused conversation
)

pprint(response)

{'email': "Hi Seán, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='ebc4497a-d77c-4b07-a52f-49a6e12f579f'),
              AIMessage(content="I can help you with that, but I need to know the sender's email address of the original message so I can locate it. Could you please provide me with this information?", additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-07-16T16:31:14.461152Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2497647292, 'load_duration': 166121292, 'prompt_eval_count': 203, 'prompt_eval_duration': 798513000, 'eval_count': 37, 'eval_duration': 1500352000, 'logprobs': None, 'model_name': 'qwen2.5:3b', 'model_provider': 'ollama'}, id='lc_run--019f6bc4-9039-71e3-9242-665bc24d2928-0', tool_calls=[], invalid_too

# Reject

In [10]:
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "reject",
                    # An explanation of why the request was rejected
                    "message": "No please sign off - Your merciful leader, Seán."
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)

{'email': "Hi Seán, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='ebc4497a-d77c-4b07-a52f-49a6e12f579f'),
              AIMessage(content="I can help you with that, but I need to know the sender's email address of the original message so I can locate it. Could you please provide me with this information?", additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-07-16T16:31:14.461152Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2497647292, 'load_duration': 166121292, 'prompt_eval_count': 203, 'prompt_eval_duration': 798513000, 'eval_count': 37, 'eval_duration': 1500352000, 'logprobs': None, 'model_name': 'qwen2.5:3b', 'model_provider': 'ollama'}, id='lc_run--019f6bc4-9039-71e3-9242-665bc24d2928-0', tool_calls=[], invalid_too

In [13]:
if "__interrupt__" in response:
    print(response["__interrupt__"][0].value["action_requests"][0]["args"]["body"])
else:
    print("No interrupt found")

No interrupt found


# Edit

In [14]:
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "edit",
                    # Edited action with tool name and args
                    "edited_action": {
                        # Tool name to call.
                        # Will usually be the same as the original action.
                        "name": "send_email",
                        # Arguments to pass to the tool.
                        "args": {"body": "This is the last straw, you're fired!"},
                    }
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)

{'email': "Hi Seán, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='ebc4497a-d77c-4b07-a52f-49a6e12f579f'),
              AIMessage(content="I can help you with that, but I need to know the sender's email address of the original message so I can locate it. Could you please provide me with this information?", additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-07-16T16:31:14.461152Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2497647292, 'load_duration': 166121292, 'prompt_eval_count': 203, 'prompt_eval_duration': 798513000, 'eval_count': 37, 'eval_duration': 1500352000, 'logprobs': None, 'model_name': 'qwen2.5:3b', 'model_provider': 'ollama'}, id='lc_run--019f6bc4-9039-71e3-9242-665bc24d2928-0', tool_calls=[], invalid_too